# Trace 품질 평가

- 구분: 수강생 실습본
- 원본: [`04_evaluation.py`](04_evaluation.py)
- 연결 교안: [4-2-langfuse-observability-and-evaluation.pdf](../../99_pdf/Chapter02/day02/4-2-langfuse-observability-and-evaluation.pdf)

원본 Python 코드는 수정 없이 아래의 한 코드 셀에 담았습니다. 설명과 실행 준비는 별도 셀입니다. **모두 실행** 대신 필요한 셀을 직접 선택하세요.


## 학습 목표

1. 규칙·LLM Judge·사용자 피드백 점수를 비교한다.
2. 코드를 실행하기 전 입력, 출력, 외부 호출과 파일 변경 가능성을 설명한다.
3. CodeMate가 만든 결과를 원본과 교안에 비춰 검토한다.


## 코드를 읽는 순서

세 평가 구간의 점수 생성과 trace_id 연결을 차례대로 확인한다.

교안은 개념과 목표를 설명합니다. 코드에서 실제 구현된 부분과 TODO를 구분하세요.


## 실행 준비

저장소 안에서 VS Code로 열고 프로젝트 `.venv` 커널을 선택합니다. 의존성은 `uv sync`로 준비합니다. 사내망에서는 사내 LLM 환경을, 사외에서는 허용된 provider를 준비합니다. 자격증명 값은 셀에 적지 않습니다. 아래 준비 셀은 저장소 루트를 찾고 `__file__`을 원본 Python 파일 경로로 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys
_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents) if (p / "pyproject.toml").is_file() and (p / "config" / "llm.py").is_file()), None)
if _repo is None:
    raise FileNotFoundError("저장소 안에서 노트북을 열어 주세요.")
os.chdir(_repo)
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
__file__ = str(_repo / 'Chapter02/05-observability/04_evaluation.py')
print(f"실행 기준 폴더: {_repo}")


## 원본 코드 — 직접 실행

**조건과 부작용:** LLM을 여러 번 호출하고 Trace와 Score를 Langfuse에 기록한다. 사용자 피드백은 코드의 시뮬레이션이다.

준비 셀을 먼저 실행하고 필요한 서비스·설정을 확인하세요. 다음 셀은 `manual-run`으로 표시했습니다. 원본 전체가 한 번에 실행되므로 API 요청, 파일 변경 또는 대기가 생길 수 있습니다.


In [ ]:
"""
평가(Evaluation): LLM 응답에 점수 매기기

핵심 개념:
  - 트레이싱만으로는 '답변이 좋은지 나쁜지' 알 수 없다.
  - score()로 각 트레이스에 점수를 남기면 품질 추세를 정량적으로 추적할 수 있다.
  - 평가 방식 3가지:
      1. 수동 평가  — 사람이 직접 점수 입력
      2. 규칙 기반  — 길이·키워드 포함 여부 등 코드로 판단
      3. LLM 평가   — 다른 LLM이 응답 품질을 채점 (LLM-as-a-Judge)

실행:
  python Chapter02/05-observability/04_evaluation.py
"""

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

langfuse = get_client()
llm = get_llm()


def run_chain(question: str, handler: CallbackHandler) -> str:
    prompt = ChatPromptTemplate.from_messages([
        ('system', '한국어로 간결하게 답변하세요. 반드시 예시를 포함하세요.'),
        ('human', '{question}'),
    ])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({'question': question}, config={'callbacks': [handler]})


# ── 1. 규칙 기반 평가 ──────────────────────────────────────────────────────────
print('=== 규칙 기반 평가 ===')
handler1 = CallbackHandler()
question1 = 'Python 리스트와 튜플의 차이를 설명해줘'
answer1 = run_chain(question1, handler1)
print(f'답변: {answer1}\n')

# get_trace_id(): 직전 invoke에서 생성된 trace의 ID를 반환
trace_id1 = handler1.last_trace_id

# 예시 포함 여부를 규칙으로 판단해 점수 부여
has_example = any(keyword in answer1 for keyword in ['예:', '예를 들어', '예시', '[', '('])
langfuse.create_score(
    trace_id=trace_id1,
    name='has-example',        # 대시보드에 표시될 평가 항목 이름
    value=1.0 if has_example else 0.0,
    comment='예시 포함 여부 자동 검사',
)
print(f'예시 포함 여부: {"✓" if has_example else "✗"} (score={1.0 if has_example else 0.0})')
print()


# ── 2. LLM-as-a-Judge 평가 ────────────────────────────────────────────────────
# 별도 LLM이 응답 품질을 0~1로 채점 — 사람 평가 없이 대규모 자동 평가에 사용
print('=== LLM-as-a-Judge 평가 ===')
handler2 = CallbackHandler()
question2 = '딥러닝이란 무엇인가요?'
answer2 = run_chain(question2, handler2)
print(f'답변: {answer2}\n')

trace_id2 = handler2.last_trace_id

judge_prompt = ChatPromptTemplate.from_messages([
    ('system', '아래 답변의 품질을 0.0~1.0 사이 숫자 하나만 출력하세요. 다른 텍스트는 출력하지 마세요.'),
    ('human', '질문: {question}\n답변: {answer}'),
])
judge_chain = judge_prompt | llm | StrOutputParser()

# judge LLM 호출은 평가용이므로 별도 트레이스로 기록하지 않음 (callbacks 미전달)
score_str = judge_chain.invoke({'question': question2, 'answer': answer2})
try:
    score_value = max(0.0, min(1.0, float(score_str.strip())))
except ValueError:
    score_value = 0.5

langfuse.create_score(
    trace_id=trace_id2,
    name='llm-judge-quality',
    value=score_value,
    comment=f'LLM 자동 채점: {score_str.strip()}',
)
print(f'LLM 채점 점수: {score_value}')
print()


# ── 3. 수동 평가 시뮬레이션 ────────────────────────────────────────────────────
# 실제 서비스에서는 사용자 피드백(좋아요/싫어요)을 받아 score()로 기록
print('=== 수동 평가 시뮬레이션 ===')
handler3 = CallbackHandler()
answer3 = run_chain('강화학습을 쉽게 설명해줘', handler3)
trace_id3 = handler3.last_trace_id

# 사용자가 👍를 눌렀다고 가정
# 랜덤한 상황에 에이전트 2개를 병렬적으로 실행한 결과를 사용자에게 보여주고,
    # 그 때에만 잠시 사용자의 승인을 대기하는 방식으로 처리
    # 그때 사용자가 답변한 응답을 토대로 점수 기록
user_feedback = 1.0
langfuse.create_score(
    trace_id=trace_id3,
    name='user-feedback',
    value=user_feedback,
    comment='사용자 좋아요',
)
print(f'답변: {answer3}')
print(f'사용자 평가: 👍 (score={user_feedback})')
print()

langfuse.flush()
print('✓ 대시보드 [Scores] 탭에서 평가 결과를 확인하세요.')


## CodeMate에게 요청하고 검토하기

> 세 평가 방식의 기준, 비용, 한계를 표로 정리하고 각 Score가 어떤 Trace에 연결되는지 검토해 줘.

CodeMate가 제안한 수정의 근거를 읽고, 실행 가능한 환경에서 출력·Trace·생성 파일을 확인하세요.


## 확인 질문

1. 예시 단어 포함만으로 품질을 판단할 수 있는가? 사용자 피드백은 실제 입력인가?
2. 실행 전 어떤 환경과 자료를 확인해야 하는가?
3. 결과가 기대와 다르면 어느 단계의 기록부터 볼 것인가?
